# Lección 9.9 — Recetario de diagnóstico: el consultorio de redes neuronales

> **Módulo 9 · Redes neuronales desde la intuición** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 110–130 min (lección con ejercicios intercalados ≈ 80 min · ejercicios finales ≈ 35 min)
> 📋 Requisitos previos: Lección 3.7 (curvas de aprendizaje), Lección 9.3 (activaciones y funciones de pérdida), Lección 9.4 (bucle de entrenamiento), Lecciones 9.5–9.8

### 🎯 Objetivos
Al terminar esta lección podrás:
- Leer las curvas y los números de un entrenamiento como un médico lee los signos vitales.
- Hacer dos **pruebas de cordura** antes de entrenar en serio: la **pérdida inicial esperada** y **sobreajustar un solo lote**.
- Diagnosticar los tres síntomas clásicos: **"mi pérdida no baja"**, **"mi pérdida es NaN"** y **"valida peor que entrena"**.
- Reconocer las causas más frecuentes (tasa de aprendizaje, datos sin normalizar, etiquetas desalineadas, softmax doble, `zero_grad` olvidado, `eval()` olvidado, sobreajuste) y saber qué probar en cada caso.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. En esta lección vas a ser **médico de redes**: los pacientes llegan con un entrenamiento roto a propósito y tú tienes que encontrar la causa **antes** de mirar la respuesta.

## ▶️ Preparación
Usaremos MNIST (dígitos de 28×28): 3,000 imágenes para entrenar y 2,000 para validar. Las imágenes se guardan **sin normalizar** (píxeles de 0 a 255), y el laboratorio decide si normalizarlas.

In [ ]:
import json, base64, math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import datasets

torch.manual_seed(0)
rng = np.random.default_rng(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
plt.rcParams["figure.figsize"] = (8, 4.5)

mnist = datasets.MNIST("data", train=True, download=True)
orden = torch.randperm(60_000, generator=torch.Generator().manual_seed(0))
X = mnist.data.float().view(-1, 28 * 28)          # píxeles de 0 a 255, aplanados
y = mnist.targets
X_tr, y_tr = X[orden[:3000]], y[orden[:3000]]
X_va, y_va = X[orden[3000:5000]], y[orden[3000:5000]]

# 🙈 Las fallas de los "pacientes" están escondidas a propósito. ¡No las decodifiques antes de diagnosticar!
PACIENTES = json.loads(base64.b64decode(
    "eyJBIjogeyJvcHQiOiAiYWRhbSIsICJsciI6IDEuMH0sICJCIjogeyJvcHQiOiAic2dkIiwgImxyIjogMC4wMDAxfSwgIkMiOiB7Im5vcm1hbGl6YXIiOiBmYWxzZSwgIm9wdCI6ICJzZ2QiLCAibHIiOiAwLjAxfSwgIkQiOiB7InNvZnRtYXhfZG9ibGUiOiB0cnVlfSwgIkUiOiB7InBlcmRpZGFfY2FzZXJhIjogdHJ1ZSwgImxyIjogMC4xfSwgIkYiOiB7Inplcm9fZ3JhZCI6IGZhbHNlLCAib3B0IjogInNnZCIsICJsciI6IDAuMDV9LCAiRyI6IHsiYmFyYWphcl9ldGlxdWV0YXMiOiB0cnVlfSwgIlgiOiB7Im5fdHJhaW4iOiAzMDAsICJvY3VsdGFzIjogNTEyLCAiZXBvY2FzIjogNDB9LCAiWSI6IHsibm9ybWFsaXphciI6IGZhbHNlLCAib3B0IjogInNnZCIsICJsciI6IDAuMDAxfSwgIloiOiB7InNvZnRtYXhfZG9ibGUiOiB0cnVlLCAib3B0IjogInNnZCIsICJsciI6IDAuMDV9fQ=="))
print(f"✅ Todo listo · {len(X_tr)} imágenes de entrenamiento y {len(X_va)} de validación · dispositivo: {device}")

### ▶️ El laboratorio
Esta función entrena una red sencilla (784 → 128 → 10, ReLU) con el bucle de la Lección 9.4. Cada argumento es una **perilla** que, mal puesta, rompe el entrenamiento de una forma distinta. Con los valores por defecto el entrenamiento es **sano**. No necesitas entender cada línea: solo fíjate en los nombres de los argumentos.

In [ ]:
def entrenar(lr=1e-3, opt="adam", normalizar=True, zero_grad=True, barajar_etiquetas=False,
             softmax_doble=False, perdida_casera=False, n_train=3000, ocultas=128, epocas=15):
    torch.manual_seed(0)
    capas = [nn.Linear(784, ocultas), nn.ReLU(), nn.Linear(ocultas, 10)]
    if softmax_doble:
        capas.append(nn.Softmax(dim=1))                       # ¡CrossEntropyLoss ya aplica softmax!
    red = nn.Sequential(*capas)
    optimizador = (torch.optim.Adam if opt == "adam" else torch.optim.SGD)(red.parameters(), lr=lr)

    xt, xv = X_tr[:n_train], X_va
    if normalizar:
        xt, xv = xt / 255, xv / 255                            # píxeles entre 0 y 1
    yt = y_tr[:n_train]
    if barajar_etiquetas:
        yt = yt[torch.randperm(n_train)]                       # las etiquetas ya no corresponden a su imagen

    def perdida_fn(salida, objetivo):
        if perdida_casera:                                     # entropía cruzada "hecha a mano"
            return -torch.log(torch.softmax(salida, 1)[range(len(objetivo)), objetivo]).mean()
        return F.cross_entropy(salida, objetivo)

    h = {"train": [], "val": [], "acc_train": [], "acc_val": [], "primer_lote": None, "pico": 0.0}
    for _ in range(epocas):
        red.train()
        orden_lotes = torch.randperm(n_train)
        for i in range(0, n_train, 64):
            idx = orden_lotes[i:i + 64]
            if zero_grad:
                optimizador.zero_grad()
            perdida = perdida_fn(red(xt[idx]), yt[idx])
            if h["primer_lote"] is None:
                h["primer_lote"] = perdida.item()
            if len(h["train"]) == 0:                           # pérdida más alta vista durante la 1.ª época
                h["pico"] = max(h["pico"], perdida.item()) if not math.isnan(perdida.item()) else float("nan")
            perdida.backward()
            optimizador.step()
        red.eval()
        with torch.no_grad():
            for nombre, xs, ys in [("train", xt, yt), ("val", xv, y_va)]:
                salida = red(xs)
                h[nombre].append(perdida_fn(salida, ys).item())
                h["acc_" + nombre].append((salida.argmax(1) == ys).float().mean().item())
    return h

def ver(h, titulo=""):
    e = len(h["train"])
    marcas = sorted({1, 5, 10, e} & set(range(1, e + 1)))
    print(f"Pérdida del primer lote: {h['primer_lote']:.3g}   (referencia ln(10) = {math.log(10):.2f})")
    print(f"Pérdida más alta durante la 1.ª época: {h['pico']:.3g}")
    print("Época            :", "  ".join(f"{m:>7}" for m in marcas))
    print("Pérdida train    :", "  ".join(f"{h['train'][m - 1]:>7.3g}" for m in marcas))
    print("Pérdida val      :", "  ".join(f"{h['val'][m - 1]:>7.3g}" for m in marcas))
    print(f"Exactitud final  : train {h['acc_train'][-1]:.0%} · val {h['acc_val'][-1]:.0%}")
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
    axes[0].plot(range(1, e + 1), h["train"], "o-", ms=3, label="train")
    axes[0].plot(range(1, e + 1), h["val"], "s-", ms=3, label="validación")
    axes[0].axhline(math.log(10), ls="--", c="gray", label="ln(10) ≈ 2.30")
    axes[0].set_title(f"Pérdida {titulo}"); axes[0].set_xlabel("época"); axes[0].legend()
    axes[1].plot(range(1, e + 1), h["acc_train"], "o-", ms=3, label="train")
    axes[1].plot(range(1, e + 1), h["acc_val"], "s-", ms=3, label="validación")
    axes[1].set_ylim(0, 1.02); axes[1].set_title(f"Exactitud {titulo}"); axes[1].set_xlabel("época"); axes[1].legend()
    plt.tight_layout(); plt.show()

def paciente(letra):
    return entrenar(**PACIENTES[letra])

---
## 1. 📖 La intuición: síntoma no es lo mismo que causa

Vas al médico con fiebre. La fiebre es un **síntoma**: puede venir de una gripe, de una infección o de un golpe de calor. Un buen médico no receta lo primero que se le ocurre; **observa** los signos vitales, **descarta** causas una por una y **cambia una sola cosa a la vez** para ver si funcionó.

Con las redes pasa igual. "Mi red no aprende" es la fiebre. Las causas posibles son muchas, y cada una deja **huellas distintas** en las curvas:
- ¿Dónde **empieza** la pérdida?
- ¿**Baja**, se queda plana, salta, o se vuelve `nan`?
- ¿Cómo se comparan **entrenamiento** y **validación**?

> 🩺 Regla de oro del consultorio: **cambia una sola cosa a la vez.** Si cambias la tasa de aprendizaje, el optimizador y la arquitectura al mismo tiempo y mejora, no sabrás qué lo arregló (ni qué romperá la próxima vez).

### ✋ Ejercicio 1 — El médico apresurado
Tu red da 60 % de exactitud. Un compañero, en un solo experimento, cambia el optimizador, duplica las capas, sube la tasa de aprendizaje y añade dropout. Ahora da 85 %. ¿Qué problema tiene su forma de trabajar?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

No sabe **cuál** de los cuatro cambios ayudó; quizá uno ayudó mucho y otro estorbó (y con solo ese cambio llegaría al 90 %). Tampoco sabrá qué hacer cuando el problema vuelva a aparecer en otro proyecto. Es como tomar cuatro medicinas a la vez: te curas, pero no sabes de qué estabas enfermo.
</details>

---
## 2. 📖 Los signos vitales de una red sana

Antes de los pacientes, conozcamos a una persona **sana**. Hay dos pruebas de cordura que conviene hacer **siempre**, antes de gastar horas entrenando.

### Prueba 1: la pérdida inicial esperada
Una red recién creada no sabe nada: reparte la probabilidad más o menos **por igual** entre las clases. Con 10 clases le da ≈1/10 a la correcta, y la entropía cruzada de eso es **−ln(1/10) = ln(10) ≈ 2.30**. Si la pérdida del **primer lote** está muy lejos de ese valor, algo va mal **antes** de empezar.

### 🔮 Ejercicio 2 — ¿Cuánto debería valer?
¿Qué pérdida inicial esperas en un problema de **2 clases**? ¿Y de **100 clases**? Calcula y comprueba.

✍️ *Mis cálculos:*

In [ ]:
for n_clases in [2, 10, 100]:     # ✏️ prueba otros valores
    print(f"{n_clases:>3} clases → pérdida inicial esperada ln({n_clases}) = {math.log(n_clases):.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

2 clases → **0.69** · 10 clases → **2.30** · 100 clases → **4.61**. Memoriza al menos el de 10: es la "temperatura normal" de casi todos los ejemplos de este curso. Si una red se queda **clavada** en ese valor, está adivinando al azar.
</details>

### ▶️ La red sana
Ahora entrena al paciente sano y mira sus signos vitales. Fíjate en tres cosas: dónde empieza la pérdida, cómo baja y qué distancia hay entre entrenamiento y validación.

In [ ]:
h_sana = entrenar()
ver(h_sana, "(red sana)")

<details><summary><b>👀 Qué deberías ver</b></summary>

- Primer lote: ≈2.3, justo lo esperado.
- La pérdida de entrenamiento baja rápido al principio y luego más despacio, hasta ≈0.07; la de validación baja y se estabiliza alrededor de ≈0.30.
- Exactitud final: ≈99 % en entrenamiento y ≈92 % en validación. Hay una brecha (normal con solo 3,000 ejemplos), pero la validación **no empeora**.

Guarda esta imagen en la cabeza: es la referencia contra la que compararás a cada paciente.
</details>

### Prueba 2: ¿puede memorizar un solo lote?
Una red sana, con 64 ejemplos y suficientes pasos, debe ser capaz de **memorizarlos** (pérdida casi 0, exactitud 100 %). Si **ni siquiera** puede eso, el problema no es de generalización: hay un error en el código, en los datos o en la pérdida. Es la prueba más barata y más útil del recetario.

### 🔮 Ejercicio 3 — Memorizar 64 ejemplos
Predice: tras 200 pasos sobre **el mismo lote** de 64 imágenes, ¿qué pérdida y qué exactitud tendrá la red sana? ¿Y si las etiquetas de ese lote están **barajadas** (no corresponden a sus imágenes)?

✍️ *Mi predicción:*

In [ ]:
def sobreajustar_un_lote(barajar=False, pasos=200):
    torch.manual_seed(0)
    red = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10))
    opt = torch.optim.Adam(red.parameters(), lr=1e-3)
    xb, yb = X_tr[:64] / 255, y_tr[:64]
    if barajar:
        yb = yb[torch.randperm(64)]
    for paso in range(1, pasos + 1):
        opt.zero_grad()
        perdida = F.cross_entropy(red(xb), yb)
        perdida.backward()
        opt.step()
        if paso in (1, 50, 100, pasos):
            acc = (red(xb).argmax(1) == yb).float().mean().item()
            print(f"  paso {paso:>3}: pérdida {perdida.item():.4f} · exactitud en el lote {acc:.0%}")

print("Etiquetas correctas:"); sobreajustar_un_lote()
print("Etiquetas barajadas:"); sobreajustar_un_lote(barajar=True)

<details><summary><b>👀 Ver respuesta</b></summary>

- **Etiquetas correctas:** la pérdida cae a casi 0 (≈0.003 en el paso 200) y la exactitud llega al 100 %. ✅ El código, los datos y la pérdida funcionan.
- **Etiquetas barajadas:** ¡**también** llega a 100 % en el paso 100, con pérdida ≈0.005 en el paso 200! Con 64 ejemplos y 100,000 pesos, la red puede memorizar cualquier cosa, incluso etiquetas sin sentido.

Moraleja: pasar esta prueba es **necesario pero no suficiente**. Si **no** la pasas, tienes un error seguro y debes arreglarlo antes de seguir. Si la pasas, aún tienes que mirar la validación.
</details>

---
## 3. 🩺 Síntoma 1: "mi pérdida no baja"

Llegan cuatro pacientes con la misma queja. Para cada uno: ejecuta, mira los signos vitales, escribe tu diagnóstico **y después** abre la respuesta. Ten a mano esta lista de sospechosos:

| Sospechoso | Huella típica |
|---|---|
| Tasa de aprendizaje **demasiado alta** | La pérdida salta a valores enormes en los primeros pasos y luego se queda plana o rebota |
| Tasa de aprendizaje **demasiado baja** | La pérdida baja, pero tan despacio que parece plana |
| Entradas **sin normalizar** | Pérdida del primer lote **muy por encima** de ln(n_clases) |
| **Softmax doble** (softmax en la red + `CrossEntropyLoss`) | La pérdida baja un poco y se estanca en un "piso" alto que nunca cruza |

### 🔮 Paciente A

In [ ]:
ver(paciente("A"), "(paciente A)")

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver diagnóstico</b></summary>

**Tasa de aprendizaje demasiado alta** (Adam con lr = 1). La pista está en la segunda línea: durante la primera época la pérdida llegó a **valores enormes** (el "pico", muy por encima de 2.30). Los primeros pasos lanzaron los pesos tan lejos que la red quedó "muerta": muchas ReLU se apagaron y, desde la época 1, la pérdida queda **plana en ≈2.3** y la exactitud en ≈12 %, que es adivinar. **Receta:** bajar la tasa ×100–×1000 (Adam con 1e-3). Si no sabes por dónde empezar, haz un *LR range test* (Lección 9.6).
</details>

### 🔮 Paciente B

In [ ]:
ver(paciente("B"), "(paciente B)")

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver diagnóstico</b></summary>

**Tasa de aprendizaje demasiado baja** (SGD con lr = 0.0001). Empieza en ≈2.3 como debe, no hay picos, y la pérdida baja… de 2.30 a 2.29 en 15 épocas. La red sí aprende (exactitud ≈17 %, algo mejor que el azar), pero a paso de tortuga. **Receta:** subir la tasa ×100 o más (con SGD, lr = 0.05–0.1), o usar Adam con 1e-3.

Compara con A: **las dos curvas terminan planas cerca de 2.30**. Lo que las distingue es el pico de la primera época. Por eso conviene mirar la pérdida **lote a lote** al principio, no solo por época.
</details>

### 🔮 Paciente C
Pista: no te quedes solo con las curvas; mira con lupa la **primera línea** de la salida.

In [ ]:
ver(paciente("C"), "(paciente C)")

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver diagnóstico</b></summary>

**Entradas sin normalizar** (píxeles de 0 a 255). La primera línea lo delata: la pérdida del primer lote es ≈**22.6**, casi diez veces el 2.30 esperado. Con entradas enormes, las sumas ponderadas son enormes, la red arranca "sobreconfiada" y sus primeros pasos son violentos. Después se recupera a medias: la pérdida queda en ≈1.9 y la exactitud en ≈29 %. **Receta:** escalar las entradas (dividir entre 255 o estandarizar, Lección 2.3).
</details>

### 🔮 Paciente D

In [ ]:
ver(paciente("D"), "(paciente D)")

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver diagnóstico</b></summary>

**Softmax doble.** Empieza bien (≈2.30), baja en las primeras épocas y luego se **estanca en un piso** (≈1.6) que nunca cruza, aunque la exactitud no es mala (≈83 % en validación). La red termina en `nn.Softmax` y `CrossEntropyLoss` aplica **otro** softmax: las "puntuaciones" que recibe la pérdida están atrapadas entre 0 y 1, así que ni una predicción perfecta puede bajar la pérdida de ≈1.46 (= ln(1 + 9/e)). Además, los gradientes se encogen y el aprendizaje se frena. **Receta:** quitar el `Softmax` de la red y pasar los **logits crudos** a la pérdida (Lección 9.3).
</details>

### ✏️ Ejercicio 4 — Receta y control
Elige **uno** de los cuatro pacientes y "cúralo": escribe abajo la configuración corregida (cambiando **una sola perilla** respecto a la del paciente) y comprueba que sus signos vitales se parezcan a los de la red sana. Para ver qué perillas tiene el paciente, ahora sí puedes imprimir su historia clínica.

In [ ]:
print("Historia clínica del paciente C:", PACIENTES["C"])   # ✏️ cambia la letra

# ✏️ Escribe la receta: la misma configuración con UNA perilla corregida
ver(entrenar(normalizar=True, opt="sgd", lr=0.01), "(paciente C tratado)")

<details><summary><b>👀 Ver respuesta</b></summary>

Para el paciente C, `normalizar=True` (dejando SGD con lr = 0.01) da un primer lote de ≈2.28, la pérdida baja de forma continua (≈0.72 en la época 15) y la validación llega a ≈83 %. Todavía no alcanza a la red sana (que usa Adam), pero sus signos vitales ya son **sanos**: solo va más lento. Mejorarlo sería **otro** experimento (subir la tasa o cambiar a Adam), una cosa a la vez.

Las demás recetas de una perilla: **A** → `lr=1e-3`; **B** → `lr=0.1` (o `opt="adam", lr=1e-3`, aunque eso son dos cambios); **D** → `softmax_doble=False`.
</details>

---
## 4. 🩺 Síntoma 2: "mi pérdida es NaN"

`nan` (*not a number*) aparece cuando un cálculo produce algo indefinido, como `0 × ∞` o `∞ − ∞`. Basta **un** `nan` en un lote para contagiar a todos los pesos en el siguiente paso, y a partir de ahí la red está muerta.

Las dos causas más comunes son: **pasos gigantes** (tasa de aprendizaje alta, gradientes que explotan, Lección 9.7) y **operaciones peligrosas** como el logaritmo de 0. Mira esta última a mano:

In [ ]:
salida = torch.tensor([[0.0, 120.0]])               # la red está MUY segura de la clase 1…
probabilidades = torch.softmax(salida, dim=1)
print("probabilidades           :", probabilidades)
print("log de las probabilidades:", torch.log(probabilidades))
print("…pero la respuesta correcta era la clase 0")
print("pérdida 'casera'  -log(p):", -torch.log(probabilidades[0, 0]).item())
print("F.cross_entropy           :", F.cross_entropy(salida, torch.tensor([0])).item())

### ✋ Ejercicio 5 — ¿Por qué una da `inf` y la otra no?
La probabilidad de la clase 0 es tan pequeña (e⁻¹²⁰) que la computadora la **redondea a 0**, y log(0) = −∞. ¿Por qué `F.cross_entropy` sí da un número finito con la misma salida?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Porque `F.cross_entropy` **nunca calcula la probabilidad por separado**: combina softmax y logaritmo en una sola fórmula estable (*log-softmax*), que trabaja directamente con las puntuaciones y da **120**, el valor correcto. La versión casera da `inf`, y en el siguiente `backward()` el `inf` se convierte en `nan`.

👉 Usa siempre las pérdidas de PyTorch (`CrossEntropyLoss`, `BCEWithLogitsLoss`) sobre las **puntuaciones crudas** (*logits*), como en la Lección 9.3.
</details>

### 🔮 Paciente E

In [ ]:
ver(paciente("E"), "(paciente E)")

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver diagnóstico</b></summary>

**Pérdida casera con log(0).** Primer lote normal (≈2.28), pero desde la primera época todo es `nan` y la exactitud se queda en ≈10 %. La configuración usa la entropía cruzada "hecha a mano" con una tasa alta (lr = 0.1): en cuanto la red se vuelve muy segura de una respuesta equivocada, la probabilidad de la clase correcta se redondea a 0, el log da −∞, y el `nan` contagia a todos los pesos. **Receta:** usar `F.cross_entropy` (con la misma tasa, la red entrena sin `nan`) y, de paso, revisar la tasa de aprendizaje.
</details>

### 🔮 Paciente F
Este paciente no llega con `nan`, pero sí con un comportamiento errático. Fíjate en cómo se mueve la pérdida de **validación**.

In [ ]:
ver(paciente("F"), "(paciente F)")

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver diagnóstico</b></summary>

**`zero_grad()` olvidado.** Fíjate en la forma: la pérdida de entrenamiento baja al principio (≈1.38 → ≈0.88) y luego **vuelve a subir** (≈1.4), mientras la de validación se dispara (≈1.6 → ≈5.1). Sin `zero_grad()`, cada `backward()` **suma** su gradiente al de todos los pasos anteriores (Lección 9.4): los pasos son cada vez más largos y apuntan a direcciones viejas. Exactitud final ≈56 % en validación. **Receta:** `optimizador.zero_grad()` al inicio de cada paso.
</details>

---
## 5. 🩺 Síntoma 3: "valida peor que entrena"

Que la validación sea **un poco** peor que el entrenamiento es normal. Lo preocupante es una brecha **grande** o una validación que **empeora** con las épocas. Las causas principales: **sobreajuste** (Lección 9.8), **datos que no corresponden** (etiquetas mal alineadas, fuga de datos, Lección 2.6, o validación de otra distribución) y **evaluar en modo entrenamiento**.

### 🔮 Paciente G

In [ ]:
ver(paciente("G"), "(paciente G)")

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver diagnóstico</b></summary>

**Etiquetas desalineadas** (las etiquetas de entrenamiento se barajaron y ya no corresponden a sus imágenes). La huella: el entrenamiento mejora **muy despacio** (pérdida 2.28 → 1.69, exactitud ≈46 %) porque la red está **memorizando ruido**, mientras la validación **empeora** (2.33 → 2.59) y su exactitud es ≈9 %: **el nivel del azar**. Un sobreajuste normal deja la validación por debajo del entrenamiento, pero nunca al nivel del azar. **Receta:** revisar cómo se unieron datos y etiquetas (un `shuffle` aplicado solo a una de las dos, un `merge` por la columna equivocada…) y mirar ejemplos con su etiqueta.
</details>

### 🔮 Paciente X

In [ ]:
ver(paciente("X"), "(paciente X)")

✍️ *Mi diagnóstico:*

<details><summary><b>👀 Ver diagnóstico</b></summary>

**Sobreajuste** de manual (Lección 9.8). Esta configuración entrena una red grande (512 neuronas) con solo **300 imágenes** durante 40 épocas. El entrenamiento llega a pérdida ≈0.01 y 100 % de exactitud; la pérdida de validación toca fondo hacia la época 10 (≈0.56) y luego **sube** (≈0.61 en la época 40), con 83 % de exactitud. La forma de "U" en validación es la huella. **Receta:** más datos o aumento de datos, dropout, weight decay, early stopping o una red más pequeña.
</details>

### 🔮 Ejercicio 6 — El termómetro que cambia solo
Una red con **dropout** se evalúa dos veces seguidas con **los mismos datos de validación**, pero alguien olvidó llamar a `red.eval()`. ¿Qué crees que verá?

✍️ *Mi predicción:*

In [ ]:
torch.manual_seed(0)
red = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, 10))
opt = torch.optim.Adam(red.parameters(), lr=1e-3)
for _ in range(5):
    for i in range(0, 3000, 64):
        opt.zero_grad(); F.cross_entropy(red(X_tr[i:i + 64] / 255), y_tr[i:i + 64]).backward(); opt.step()

with torch.no_grad():
    for modo in ["train", "train", "eval", "eval"]:
        red.train() if modo == "train" else red.eval()
        acc = (red(X_va / 255).argmax(1) == y_va).float().mean().item()
        print(f"modo {modo:<5} → exactitud en validación: {acc:.2%}")

<details><summary><b>👀 Ver respuesta</b></summary>

En modo `train` la exactitud **cambia** de una evaluación a otra (≈86.9 % y ≈86.3 %), porque cada vez el dropout apaga neuronas distintas, y además sale **más baja**. En modo `eval` da ≈89.3 % las dos veces. Si olvidas `red.eval()`, tus métricas serán ruidosas y pesimistas, y podrías descartar una buena red o comparar modelos de forma injusta (Lección 9.8).
</details>

---
## 6. 📋 El recetario completo

| 🚨 Síntoma | Huella en las curvas | Causa probable | Qué probar primero |
|---|---|---|---|
| **No baja** | Plana en ≈ln(n_clases) desde el principio | lr demasiado baja · gradientes que desaparecen (9.7) · red "muerta" tras un lr alto | Subir lr ×10 · ReLU + He · revisar inicialización |
| **No baja** | Primer lote ≫ ln(n_clases) | Entradas sin normalizar | Escalar entradas (2.3) |
| **No baja** | Salta a valores enormes y luego queda plana | lr demasiado alta | Bajar lr ×10, *LR range test* (9.6), warmup |
| **No baja** | Baja y se estanca en un "piso" alto | Softmax doble · activación de salida incorrecta (9.3) | Quitar softmax, pasar logits crudos a la pérdida |
| **No baja** (ni con un lote) | No memoriza 64 ejemplos | Error de código: pérdida equivocada, datos y etiquetas desalineados, `step()` olvidado | Prueba "sobreajustar un lote" e imprimir ejemplos con su etiqueta |
| **NaN / inf** | De golpe, a veces tras unas épocas | log(0) casero · lr alta · gradientes que explotan · división por cero | Pérdidas de PyTorch sobre logits · bajar lr · `clip_grad_norm_` |
| **Errática** | Sube y baja sin control, empeora | `zero_grad()` olvidado · lr alta · lotes muy pequeños | Revisar el bucle (9.4) · bajar lr · subir batch size |
| **Valida peor que entrena** | Train → 0, val sube (forma de "U") | Sobreajuste | Más datos/aumento, dropout, weight decay, early stopping (9.8) |
| **Valida peor que entrena** | Train baja lento, val **peor que el azar** | Etiquetas desalineadas o validación de otra distribución | Revisar cómo se unieron datos y etiquetas |
| **Validación inestable** | El mismo modelo da números distintos | `eval()` olvidado con dropout/BatchNorm | `red.eval()` al medir, `red.train()` al entrenar |
| **Sospechosamente perfecto** | ~100 % en validación en un problema difícil | Fuga de datos (2.6) | Buscar columnas o ejemplos que "contienen" la respuesta |

### ✅ Checklist antes de entrenar en serio
1. Mira **unos cuantos ejemplos con su etiqueta**. ¿Coinciden?
2. ¿Las entradas están **normalizadas**? (media ≈ 0, desviación ≈ 1, o al menos entre 0 y 1).
3. ¿La pérdida del primer lote es ≈ **ln(n_clases)**?
4. ¿La red **sobreajusta un solo lote**?
5. ¿La salida de la red son **logits** y la pérdida es la de PyTorch adecuada?
6. ¿El bucle tiene `zero_grad()` → `backward()` → `step()`, y `train()`/`eval()` en su sitio?
7. Empieza con **Adam, lr = 1e-3** y una red pequeña. Complica solo cuando lo simple funcione.

## 7. 🧠 Resumen en 5 puntos
1. Síntoma no es causa: observa los signos vitales y **cambia una sola cosa a la vez**.
2. Dos pruebas de cordura baratísimas: pérdida inicial ≈ **ln(n_clases)** y **sobreajustar un lote**.
3. "No baja": sospecha de la **tasa de aprendizaje**, las **entradas sin normalizar** y el **softmax doble**.
4. "NaN": sospecha de **pasos gigantes** y de **logaritmos caseros**; usa las pérdidas de PyTorch sobre logits.
5. "Valida peor que entrena": distingue **sobreajuste** (U en validación) de **datos mal alineados** (validación al nivel del azar) y de **`eval()` olvidado** (números que cambian).

---
## 8. 🏋️ Ejercicios finales

Ya practicaste cada diagnóstico por separado. Ahora toca **combinarlos** (≈35 min).

### 8.1 Repaso rápido (opción múltiple)

**1.** Clasificador de **5 clases**. La pérdida del primer lote es 1.61. ¿Qué concluyes?
- a) Las entradas no están normalizadas · b) Es lo esperado: ln(5) ≈ 1.61 · c) La tasa es muy alta · d) Hay softmax doble

**2.** Tu red **no consigue** memorizar un lote de 32 ejemplos tras 500 pasos. ¿Qué es lo **menos** probable?
- a) Un error en la pérdida · b) Imágenes y etiquetas desalineadas · c) Sobreajuste · d) Olvidaste `opt.step()`

**3.** La pérdida va bien durante 3 épocas y de repente se vuelve `nan`. ¿Qué pruebas primero?
- a) Añadir dropout · b) Bajar la tasa de aprendizaje y revisar si hay un log casero · c) Más épocas · d) Más neuronas

**4.** Entrenamiento 100 %, validación 9 % (10 clases). ¿Diagnóstico más probable?
- a) Sobreajuste normal · b) Etiquetas de entrenamiento desalineadas con sus imágenes · c) lr baja · d) Falta BatchNorm

**5.** ¿Dónde va `zero_grad()` en el bucle?
- a) Después de `step()` y antes del siguiente `backward()` (o al inicio de cada paso) · b) Solo una vez antes del bucle · c) Después de `backward()` y antes de `step()` · d) No hace falta con Adam

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Es la "temperatura normal" para 5 clases.
2. **c)** Sobreajustar es justo lo que **queremos** en esta prueba; no lograrlo indica un error de código o de datos.
3. **b)** `nan` repentino = pasos gigantes u operación peligrosa.
4. **b)** Un sobreajuste fuerte deja la validación **por debajo** del entrenamiento, pero no al nivel del azar (10 %). Que memorice el entrenamiento y no aprenda **nada** generalizable sugiere que las etiquetas no corresponden a sus imágenes (como el paciente G).
5. **a)** En c) borrarías el gradiente recién calculado y `step()` no haría nada.
</details>

### 8.2 Preguntas para pensar (sin ejecutar)

**A.** El paciente A (tasa altísima) terminó con la pérdida plana en ≈2.30, igual que el paciente B (tasa bajísima). Si solo vieras la curva final, ¿cómo los distinguirías?

**B.** ¿Por qué la prueba "sobreajustar un lote" se hace con **un** lote y no con todo el conjunto de entrenamiento?

**C. Caso 💼.** Tu jefa te dice: *"El modelo de ayer daba 92 % y hoy, sin tocar nada, da 10 %. Alguien cambió el código de carga de datos."* ¿Cuáles son tus tres primeras comprobaciones, en orden?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Por los **primeros pasos**: el paciente A tiene picos enormes de pérdida al principio (la red "salió disparada" y quedó muerta); el B empieza en 2.30 y apenas se mueve. Además, el B mejora un poquito si le das más épocas; el A no. Por eso conviene registrar la pérdida **lote a lote** al principio, no solo por época.

**B.** Porque es **rápida** (segundos) y porque su resultado es **inequívoco**: cualquier red razonable puede memorizar 64 ejemplos. Con todo el conjunto, no lograr memorizar podría deberse a falta de capacidad o de épocas, y no sabrías si hay un error.

**C.** Una propuesta: (1) **mirar ejemplos con su etiqueta** (¿se desalinearon imágenes y etiquetas, como el paciente G? 10 % es exactamente el azar con 10 clases); (2) revisar la **normalización** (¿llegan píxeles 0–255 en vez de 0–1?, mirar la pérdida del primer lote); (3) comprobar que el orden de las **clases** (el diccionario etiqueta → número) no cambió. Todas son baratas y apuntan a "cambió la carga de datos".
</details>

### 8.3 🧩 Práctica integradora: "la red de tu colega"

Un colega te pasa su código: *"No sé qué le pasa, no aprende nada"*. Su bucle está escrito abajo **completo**. Lo diagnosticarás y lo arreglarás en 4 pasos. **No avances sin revisar cada respuesta.**

**Paso 1 ▶️ — Ejecuta su código tal cual.**

In [ ]:
def red_del_colega(arreglo_1=False, arreglo_2=False, epocas=10):
    torch.manual_seed(0)
    red = nn.Sequential(nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 10), nn.Softmax(dim=1))
    if arreglo_2:
        red = nn.Sequential(*list(red)[:-1])                 # ✏️ arreglo 2 (pista: ¿qué capa quitamos?)
    opt = torch.optim.SGD(red.parameters(), lr=0.05)
    xt, xv = X_tr, X_va
    if arreglo_1:
        xt, xv = X_tr / 255, X_va / 255                      # ✏️ arreglo 1
    primer = None
    for epoca in range(epocas):
        for i in range(0, 3000, 64):
            opt.zero_grad()
            perdida = F.cross_entropy(red(xt[i:i + 64]), y_tr[i:i + 64])
            primer = perdida.item() if primer is None else primer
            perdida.backward()
            opt.step()
    with torch.no_grad():
        acc = (red(xv).argmax(1) == y_va).float().mean().item()
    print(f"primer lote: {primer:.3f} · último lote: {perdida.item():.3f} · exactitud en validación: {acc:.1%}")

red_del_colega()

✍️ *¿Qué ves? ¿Cuál es tu primera sospecha?*

<details><summary><b>👀 Ver respuesta</b></summary>

Exactitud ≈40 % y última pérdida ≈2.0: **no baja** como debería. Curiosamente, la pérdida del primer lote (≈2.34) parece **normal**… y eso despista. Con este síntoma, los sospechosos son la tasa, la normalización y el softmax doble. Toca leer el código.
</details>

**Paso 2 ✋ — Lee el código como médico.** Repasa la checklist de la sección 6 línea por línea contra el código del colega. Encuentra **dos** fallas.

✍️ *Las dos fallas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Las entradas no se normalizan:** `xt = X_tr` usa píxeles de 0 a 255 (checklist, punto 2).
2. **Softmax doble:** la red termina en `nn.Softmax` y luego `F.cross_entropy` aplica otro softmax (checklist, punto 5).

El bucle en sí (`zero_grad` → `backward` → `step`) está bien.
</details>

**Paso 3 ✏️ — Una medicina a la vez.** Aplica **solo** el arreglo 1, luego **solo** el arreglo 2 y por último los dos juntos. ¿Cuánto aporta cada uno?

In [ ]:
red_del_colega(arreglo_1=True)                   # solo normalizar
red_del_colega(arreglo_2=True)                   # solo quitar el softmax
red_del_colega(arreglo_1=True, arreglo_2=True)   # los dos

<details><summary><b>👀 Ver respuesta</b></summary>

- **Solo normalizar:** mejora (≈53 %), pero la pérdida se queda en un piso alto (≈1.9): sigue el softmax doble.
- **Solo quitar el softmax:** ¡**empeora**! El primer lote pasa a ≈25.8 y la exactitud cae a ≈11 %. Al quitar el softmax queda **al descubierto** el problema de las entradas sin normalizar: el softmax "aplastaba" las puntuaciones enormes y escondía el síntoma de la pérdida inicial.
- **Los dos:** primer lote ≈2.29, última pérdida ≈0.34 y ≈**88.5 %** en validación. 🎉

Lección clave: **dos fallas pueden esconderse una a la otra.** Un arreglo correcto puede parecer que "empeora" las cosas cuando destapa otra falla. Por eso se va de uno en uno, pero **sin descartar** un arreglo correcto solo porque el número bajó: hay que entender *por qué* bajó.
</details>

**Paso 4 ✋ — La nota para el colega.** Escribe un mensaje breve y amable (3–4 frases) explicándole qué tenía su red, cómo lo descubriste y qué dos pruebas debería hacer siempre antes de entrenar.

✍️ *Mi mensaje:*

<details><summary><b>👀 Ver un ejemplo</b></summary>

*"Tu red tenía dos problemas. Los píxeles llegaban de 0 a 255 en lugar de 0 a 1, y la red aplicaba softmax dos veces (una al final de la red y otra dentro de `cross_entropy`). Fue difícil de ver porque un problema escondía al otro: el softmax de más disimulaba que la pérdida inicial era absurda, y solo al quitarlo apareció (≈26 en lugar de 2.30). Con los dos arreglos pasaste de ≈40 % a ≈88 %. Para la próxima: antes de entrenar comprueba que la pérdida inicial sea ≈ ln(número de clases) y que la red pueda memorizar un solo lote."*
</details>

---
## 📚 Para profundizar
- 📝 **Blog:** Andrej Karpathy, *A Recipe for Training Neural Networks* (2019): la referencia clásica de este recetario.
- 📘 **Guía:** Google, *Deep Learning Tuning Playbook* (GitHub): cómo ajustar y diagnosticar de forma sistemática.
- 📘 **Documentación:** PyTorch, `torch.autograd.detect_anomaly` (para encontrar dónde nace un `nan`).
- 🧰 Este recetario se resume también en `apoyo/diagnostico-entrenamiento.md` del curso.

---
⬅️ **Anterior:** [9.8 Regularización en redes: entrenar para el examen, no para memorizar las respuestas](9.8-regularizacion-en-redes.ipynb) · ➡️ **Siguiente:** [9.10 Proyecto del módulo: El clasificador del armario 👕](9.10-proyecto-del-modulo.md)